# VAE潜在特徴量による交差検証とConfusion Matrix

このnotebookでは、VAEの潜在特徴量を使ってSVM分類を行い、交差検証によるconfusion matrixを出力します。

In [1]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import seaborn as sns
from collections import Counter

In [2]:
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## VAEモデルの定義

まず、既存のVAEモデル構造を定義します。

In [3]:
class VAE(nn.Module):
    def __init__(self, input_size, hidden_size=256, latent_size=32):
        super(VAE, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU()
        )
        
        # 潜在変数のパラメータ
        self.fc_mu = nn.Linear(hidden_size // 2, latent_size)
        self.fc_logvar = nn.Linear(hidden_size // 2, latent_size)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, input_size),
            nn.Sigmoid()
        )
    
    def encode(self, x):
        """エンコーダーで平均と分散を計算"""
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """再パラメータ化トリック"""
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu
    
    def decode(self, z):
        """デコーダーで再構成"""
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar

## データの読み込みと前処理

In [4]:
# デバイスの設定（GPU使用可能チェック）
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用デバイス: {device}")

# データディレクトリの設定
data_dir = Path("/home/neuron/Documents/mi-analyze/data/20250512/processed")
print(f"データディレクトリ: {data_dir}")

使用デバイス: cuda
データディレクトリ: /home/neuron/Documents/mi-analyze/data/20250512/processed


In [5]:
# ウェーブレット特徴量データの読み込み
# ここではサンプルデータを使用します。実際のデータパスに応じて調整してください。

# 例: ピクルファイルからデータを読み込む
try:
    with open(data_dir / "ishii_band_powers.pkl", "rb") as f:
        data = pickle.load(f)
    print(f"データ読み込み完了: {type(data)}")
    print(f"データのキー: {data.keys() if isinstance(data, dict) else 'dict以外の形式'}")
except FileNotFoundError:
    print("サンプルデータを生成します...")
    # サンプルデータ生成（実際のデータがない場合）
    n_samples = 1000
    n_features = 256
    n_classes = 4
    
    # ランダムなEEG様データを生成
    np.random.seed(42)
    X = np.random.randn(n_samples, n_features)
    y = np.random.randint(0, n_classes, n_samples)
    
    print(f"サンプルデータ生成完了: X.shape={X.shape}, y.shape={y.shape}")
    print(f"クラス分布: {Counter(y)}")

データ読み込み完了: <class 'dict'>
データのキー: dict_keys(['band_powers', 'labels'])


## 交差検証によるSVM分類とConfusion Matrix

VAEの潜在特徴量を使って交差検証を行い、confusion matrixを生成します。

In [6]:
def extract_vae_features(model, data_loader, device):
    """VAEから潜在特徴量を抽出"""
    model.eval()
    features = []
    
    with torch.no_grad():
        for data, in data_loader:
            data = data.to(device)
            mu, logvar = model.encode(data)
            # 推論時は平均を使用（サンプリングなし）
            z = mu  # または reparameterize(mu, logvar) でサンプリング
            features.append(z.cpu().numpy())
    
    return np.vstack(features)

def cross_validation_with_confusion_matrix(X, y, model_path=None, cv_folds=5, random_state=42):
    """
    交差検証を行い、confusion matrixを生成
    
    Parameters:
    -----------
    X : numpy.ndarray
        入力データ
    y : numpy.ndarray
        ラベル
    model_path : str or None
        学習済みVAEモデルのパス（Noneの場合は新規学習）
    cv_folds : int
        交差検証のフォールド数
    random_state : int
        ランダムシード
    
    Returns:
    --------
    accuracy : float
        平均精度
    cm : numpy.ndarray
        confusion matrix
    y_pred : numpy.ndarray
        予測結果
    """
    
    print(f"交差検証開始: {cv_folds}フォールド")
    print(f"データサイズ: X={X.shape}, y={y.shape}")
    print(f"クラス分布: {Counter(y)}")
    
    # データの標準化
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # VAEモデルの準備
    input_size = X_scaled.shape[1]
    vae_model = VAE(input_size=input_size, hidden_size=256, latent_size=32).to(device)
    
    if model_path and Path(model_path).exists():
        print(f"学習済みモデルを読み込み: {model_path}")
        try:
            vae_model.load_state_dict(torch.load(model_path, map_location=device))
        except Exception as e:
            print(f"モデル読み込みエラー: {e}")
            print("新規学習を行います...")
            vae_model = train_vae_model(X_scaled, vae_model, device)
    else:
        print("新規VAEモデルを学習します...")
        vae_model = train_vae_model(X_scaled, vae_model, device)
    
    # 全データから潜在特徴量を抽出
    dataset = TensorDataset(torch.FloatTensor(X_scaled))
    data_loader = DataLoader(dataset, batch_size=64, shuffle=False)
    
    print("潜在特徴量を抽出中...")
    latent_features = extract_vae_features(vae_model, data_loader, device)
    print(f"潜在特徴量サイズ: {latent_features.shape}")
    
    # 交差検証でSVM分類
    print("交差検証によるSVM分類を実行中...")
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    svm_classifier = SVC(kernel='rbf', C=1.0, random_state=random_state)
    
    # cross_val_predictで予測結果を取得
    y_pred = cross_val_predict(svm_classifier, latent_features, y, cv=skf)
    
    # 精度計算
    accuracy = accuracy_score(y, y_pred)
    print(f"交差検証精度: {accuracy:.4f}")
    
    # Confusion Matrix計算
    cm = confusion_matrix(y, y_pred)
    
    return accuracy, cm, y_pred

def train_vae_model(X, model, device, epochs=50, lr=0.001):
    """VAEモデルを学習"""
    print("VAEモデルを学習中...")
    
    # データローダー作成
    dataset = TensorDataset(torch.FloatTensor(X))
    train_loader = DataLoader(dataset, batch_size=64, shuffle=True)
    
    # オプティマイザー
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # 学習ループ
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, (data,) in enumerate(train_loader):
            data = data.to(device)
            optimizer.zero_grad()
            
            recon_batch, mu, logvar = model(data)
            loss = vae_loss(recon_batch, data, mu, logvar)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        if (epoch + 1) % 10 == 0:
            avg_loss = total_loss / len(train_loader)
            print(f"エポック {epoch+1}/{epochs}, 損失: {avg_loss:.4f}")
    
    print("VAE学習完了")
    return model

def vae_loss(recon_x, x, mu, logvar):
    """VAE損失関数"""
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

In [7]:
# 交差検証の実行
try:
    # 学習済みモデルのパスを指定（存在する場合）
    model_path = "/home/neuron/Documents/mi-analyze/src/vae/best_vae_model.pth"
    
    accuracy, cm, y_pred = cross_validation_with_confusion_matrix(
        X, y, 
        model_path=model_path,
        cv_folds=5,
        random_state=42
    )
    
    print(f"\n=== 交差検証結果 ===")
    print(f"平均精度: {accuracy:.4f}")
    print(f"Confusion Matrix形状: {cm.shape}")
    
except Exception as e:
    print(f"エラーが発生しました: {e}")
    import traceback
    traceback.print_exc()

エラーが発生しました: name 'X' is not defined


Traceback (most recent call last):
  File "/tmp/ipykernel_95234/2418807967.py", line 7, in <module>
    X, y,
    ^
NameError: name 'X' is not defined


## Confusion Matrixの可視化

In [8]:
def plot_confusion_matrix(cm, class_names=None, normalize=False, title='Confusion Matrix'):
    """
    Confusion Matrixを可視化
    
    Parameters:
    -----------
    cm : numpy.ndarray
        confusion matrix
    class_names : list or None
        クラス名のリスト
    normalize : bool
        正規化するかどうか
    title : str
        タイトル
    """
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.2f'
        title += ' (Normalized)'
    else:
        fmt = 'd'
    
    if class_names is None:
        class_names = [f'Class {i}' for i in range(cm.shape[0])]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, 
                annot=True, 
                fmt=fmt, 
                cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names,
                cbar_kws={'label': 'Count' if not normalize else 'Proportion'})
    
    plt.title(title, fontsize=16, fontweight='bold')
    plt.xlabel('予測ラベル', fontsize=14)
    plt.ylabel('実際のラベル', fontsize=14)
    plt.tight_layout()
    plt.show()

# Confusion Matrixの可視化
if 'cm' in locals():
    # クラス名の設定（必要に応じて変更）
    class_names = [f'クラス {i}' for i in range(cm.shape[0])]
    
    # 生のカウント
    plot_confusion_matrix(cm, class_names=class_names, normalize=False, 
                         title='VAE潜在特徴量による交差検証 Confusion Matrix')
    
    # 正規化版
    plot_confusion_matrix(cm, class_names=class_names, normalize=True, 
                         title='VAE潜在特徴量による交差検証 Confusion Matrix (正規化)')

## 詳細な分類レポート

In [9]:
# 分類レポートの表示
if 'y' in locals() and 'y_pred' in locals():
    print("=== 詳細な分類レポート ===")
    class_names = [f'クラス_{i}' for i in range(len(np.unique(y)))]
    
    report = classification_report(y, y_pred, 
                                 target_names=class_names,
                                 digits=4)
    print(report)
    
    # クラス別精度の可視化
    from sklearn.metrics import precision_recall_fscore_support
    
    precision, recall, f1, support = precision_recall_fscore_support(y, y_pred)
    
    # バープロット
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    x_pos = np.arange(len(class_names))
    
    axes[0].bar(x_pos, precision, alpha=0.7, color='skyblue')
    axes[0].set_title('Precision (適合率)')
    axes[0].set_xlabel('クラス')
    axes[0].set_ylabel('値')
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(class_names, rotation=45)
    axes[0].set_ylim(0, 1)
    
    axes[1].bar(x_pos, recall, alpha=0.7, color='lightgreen')
    axes[1].set_title('Recall (再現率)')
    axes[1].set_xlabel('クラス')
    axes[1].set_ylabel('値')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(class_names, rotation=45)
    axes[1].set_ylim(0, 1)
    
    axes[2].bar(x_pos, f1, alpha=0.7, color='orange')
    axes[2].set_title('F1-Score')
    axes[2].set_xlabel('クラス')
    axes[2].set_ylabel('値')
    axes[2].set_xticks(x_pos)
    axes[2].set_xticklabels(class_names, rotation=45)
    axes[2].set_ylim(0, 1)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n各クラスのサンプル数: {support}")

## 実際のデータでの実行例

実際のEEGデータを使用する場合は、以下のように修正してください：

```python
# 実際のデータ読み込み例
with open("your_eeg_data.pkl", "rb") as f:
    eeg_data = pickle.load(f)

# データの形状を確認
X = eeg_data['features']  # 特徴量データ
y = eeg_data['labels']    # ラベルデータ

# 交差検証実行
accuracy, cm, y_pred = cross_validation_with_confusion_matrix(
    X, y, 
    model_path="path/to/your/vae_model.pth",
    cv_folds=5
)
```

## パラメータ調整

モデルの性能を向上させるために、以下のパラメータを調整できます：

1. **VAEパラメータ**:
   - `latent_size`: 潜在空間の次元（16, 32, 64など）
   - `hidden_size`: 隠れ層のサイズ
   - 学習エポック数、学習率

2. **SVMパラメータ**:
   - `kernel`: 'rbf', 'linear', 'poly'
   - `C`: 正則化パラメータ
   - `gamma`: RBFカーネルのパラメータ

3. **交差検証**:
   - `cv_folds`: フォールド数（3, 5, 10など）

In [10]:
# パラメータ調整の例
def hyperparameter_search(X, y):
    """
    グリッドサーチによるハイパーパラメータ調整
    """
    from sklearn.model_selection import GridSearchCV
    from sklearn.pipeline import Pipeline
    
    # パラメータグリッド
    param_grid = {
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
        'kernel': ['rbf', 'linear']
    }
    
    svm = SVC(random_state=42)
    grid_search = GridSearchCV(svm, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    
    # 潜在特徴量でグリッドサーチ実行
    print("ハイパーパラメータ調整中...")
    # ここでは実際の潜在特徴量が必要
    # latent_features = extract_vae_features(vae_model, data_loader, device)
    # grid_search.fit(latent_features, y)
    
    print("最適パラメータ:")
    # print(grid_search.best_params_)
    print("最高スコア:")
    # print(f"{grid_search.best_score_:.4f}")
    
    return None  # grid_search.best_estimator_

# 使用例（コメントアウト）
# best_svm = hyperparameter_search(X, y)

## 実際のEEGデータ読み込み

既存のpreprocessedデータを使用してVAE交差検証を実行します。

In [ ]:
# 実際のEEGデータを読み込む
def load_eeg_data(data_dir):
    """
    実際のEEGデータを読み込む
    """
    try:
        # エポックデータの読み込み
        epoch_files = list(data_dir.glob("*epochs*.pkl"))
        if epoch_files:
            with open(epoch_files[0], "rb") as f:
                epoch_data = pickle.load(f)
            print(f"エポックデータ読み込み: {epoch_files[0].name}")
            return epoch_data
        
        # バンドパワーデータの読み込み
        band_files = list(data_dir.glob("*band_powers*.pkl"))
        if band_files:
            with open(band_files[0], "rb") as f:
                band_data = pickle.load(f)
            print(f"バンドパワーデータ読み込み: {band_files[0].name}")
            return band_data
        
        # ウェーブレット特徴量データの読み込み
        wavelet_files = list(data_dir.glob("*wavelet*.pkl"))
        if wavelet_files:
            with open(wavelet_files[0], "rb") as f:
                wavelet_data = pickle.load(f)
            print(f"ウェーブレット特徴量データ読み込み: {wavelet_files[0].name}")
            return wavelet_data
        
        print("適切なデータファイルが見つかりません")
        return None
        
    except Exception as e:
        print(f"データ読み込みエラー: {e}")
        return None

# データ読み込み実行
print("実際のEEGデータを読み込み中...")
eeg_data = load_eeg_data(data_dir)

if eeg_data is not None:
    print(f"データタイプ: {type(eeg_data)}")
    if isinstance(eeg_data, dict):
        print(f"データキー: {list(eeg_data.keys())}")
        for key, value in eeg_data.items():
            if isinstance(value, np.ndarray):
                print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
            else:
                print(f"  {key}: {type(value)}")
else:
    print("サンプルデータを使用します")

In [ ]:
# データの前処理と形状調整
def prepare_data_for_vae(data):
    """
    VAE用にデータを前処理
    """
    if data is None:
        # サンプルデータ生成
        print("サンプルデータを生成中...")
        n_samples = 800
        n_features = 256
        n_classes = 4
        
        np.random.seed(42)
        X = np.random.randn(n_samples, n_features)
        y = np.random.randint(0, n_classes, n_samples)
        
        print(f"サンプルデータ: X.shape={X.shape}, y.shape={y.shape}")
        return X, y
    
    # 実際のデータ処理
    if isinstance(data, dict):
        # 特徴量とラベルを抽出
        possible_feature_keys = ['features', 'X', 'data', 'epochs', 'band_powers', 'wavelet_features']
        possible_label_keys = ['labels', 'y', 'targets', 'classes']
        
        X = None
        y = None
        
        # 特徴量を探す
        for key in possible_feature_keys:
            if key in data and isinstance(data[key], np.ndarray):
                X = data[key]
                print(f"特徴量として{key}を使用: shape={X.shape}")
                break
        
        # ラベルを探す
        for key in possible_label_keys:
            if key in data and isinstance(data[key], np.ndarray):
                y = data[key]
                print(f"ラベルとして{key}を使用: shape={y.shape}")
                break
        
        # 特徴量が3次元の場合（epochs x channels x features）は2次元に変換
        if X is not None and X.ndim == 3:
            print(f"3次元データを2次元に変換: {X.shape} -> ", end="")
            X = X.reshape(X.shape[0], -1)
            print(f"{X.shape}")
        
        # ラベルがない場合はダミーラベルを作成
        if y is None and X is not None:
            print("ラベルが見つからないため、ダミーラベルを作成")
            y = np.random.randint(0, 4, X.shape[0])
        
        return X, y
    
    else:
        print(f"不明なデータ形式: {type(data)}")
        return None, None

# データ準備実行
X, y = prepare_data_for_vae(eeg_data)

if X is not None and y is not None:
    print(f"\n=== 準備完了 ===")
    print(f"特徴量: {X.shape}")
    print(f"ラベル: {y.shape}")
    print(f"クラス分布: {Counter(y)}")
    print(f"特徴量の範囲: [{X.min():.3f}, {X.max():.3f}]")
else:
    print("データ準備に失敗しました")

In [ ]:
# メイン実行: VAE交差検証
if X is not None and y is not None:
    print("=== VAE交差検証を開始 ===")
    
    try:
        # 学習済みモデルのパスを指定
        model_path = "/home/neuron/Documents/mi-analyze/src/vae/best_vae_model.pth"
        
        # 交差検証実行
        accuracy, cm, y_pred = cross_validation_with_confusion_matrix(
            X, y, 
            model_path=model_path,
            cv_folds=5,
            random_state=42
        )
        
        print(f"\n=== 交差検証結果 ===")
        print(f"平均精度: {accuracy:.4f}")
        print(f"Confusion Matrix形状: {cm.shape}")
        
        # 結果をグローバル変数として保存
        globals()['final_accuracy'] = accuracy
        globals()['final_cm'] = cm
        globals()['final_y_pred'] = y_pred
        
        print("\n交差検証完了！次のセルでConfusion Matrixを可視化できます。")
        
    except Exception as e:
        print(f"エラーが発生しました: {e}")
        import traceback
        traceback.print_exc()
else:
    print("データが準備できていません。上のセルを実行してください。")

In [ ]:
# 結果の可視化
if 'final_cm' in globals():
    print("=== Confusion Matrix可視化 ===")
    
    # クラス名の設定
    n_classes = final_cm.shape[0]
    class_names = [f'クラス{i}' for i in range(n_classes)]
    
    # 生のカウント版
    plot_confusion_matrix(final_cm, 
                         class_names=class_names, 
                         normalize=False, 
                         title='VAE潜在特徴量による交差検証 Confusion Matrix')
    
    # 正規化版
    plot_confusion_matrix(final_cm, 
                         class_names=class_names, 
                         normalize=True, 
                         title='VAE潜在特徴量による交差検証 Confusion Matrix (正規化)')
    
    print(f"\n最終精度: {final_accuracy:.4f}")
else:
    print("まず上のセルで交差検証を実行してください。")

In [ ]:
# 詳細分析と結果保存
if 'final_y_pred' in globals() and 'y' in globals():
    print("=== 詳細な分類分析 ===")
    
    # 分類レポート
    class_names = [f'クラス_{i}' for i in range(len(np.unique(y)))] 
    report = classification_report(y, final_y_pred, 
                                 target_names=class_names,
                                 digits=4)
    print(report)
    
    # クラス別精度の可視化
    from sklearn.metrics import precision_recall_fscore_support
    
    precision, recall, f1, support = precision_recall_fscore_support(y, final_y_pred)
    
    # バープロット
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    x_pos = np.arange(len(class_names))
    
    axes[0].bar(x_pos, precision, alpha=0.7, color='skyblue')
    axes[0].set_title('Precision (適合率)')
    axes[0].set_xlabel('クラス')
    axes[0].set_ylabel('値')
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(class_names, rotation=45)
    axes[0].set_ylim(0, 1)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].bar(x_pos, recall, alpha=0.7, color='lightgreen')
    axes[1].set_title('Recall (再現率)')
    axes[1].set_xlabel('クラス')
    axes[1].set_ylabel('値')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(class_names, rotation=45)
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)
    
    axes[2].bar(x_pos, f1, alpha=0.7, color='orange')
    axes[2].set_title('F1-Score')
    axes[2].set_xlabel('クラス')
    axes[2].set_ylabel('値')
    axes[2].set_xticks(x_pos)
    axes[2].set_xticklabels(class_names, rotation=45)
    axes[2].set_ylim(0, 1)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n各クラスのサンプル数: {support}")
    print(f"マクロ平均 F1-Score: {np.mean(f1):.4f}")
    print(f"重み付き平均 F1-Score: {np.average(f1, weights=support):.4f}")
    
    # 結果をファイルに保存
    try:
        results = {
            'accuracy': final_accuracy,
            'confusion_matrix': final_cm,
            'y_true': y,
            'y_pred': final_y_pred,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'support': support,
            'classification_report': report
        }
        
        output_path = Path("/home/neuron/Documents/mi-analyze/src/vae/cross_validation_results.pkl")
        with open(output_path, "wb") as f:
            pickle.dump(results, f)
        print(f"\n結果を保存しました: {output_path}")
        
    except Exception as e:
        print(f"結果保存エラー: {e}")
else:
    print("まず交差検証を実行してください。")

## パラメータ比較実験

異なるVAEパラメータ（潜在次元、隠れ層サイズ）での性能比較を行います。

In [ ]:
# 異なるVAEパラメータでの比較実験
def compare_vae_parameters(X, y, param_combinations, cv_folds=3):
    """
    異なるVAEパラメータでの性能比較
    """
    results = []
    
    for params in param_combinations:
        print(f"\n=== パラメータ: {params} ===")
        
        try:
            # カスタムVAEモデルの作成
            input_size = X.shape[1]
            vae_model = VAE(
                input_size=input_size, 
                hidden_size=params['hidden_size'], 
                latent_size=params['latent_size']
            ).to(device)
            
            # データの標準化
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)
            
            # VAE学習
            vae_model = train_vae_model(
                X_scaled, vae_model, device, 
                epochs=params.get('epochs', 30),
                lr=params.get('lr', 0.001)
            )
            
            # 潜在特徴量抽出
            dataset = TensorDataset(torch.FloatTensor(X_scaled))
            data_loader = DataLoader(dataset, batch_size=64, shuffle=False)
            latent_features = extract_vae_features(vae_model, data_loader, device)
            
            # 交差検証
            skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
            svm_classifier = SVC(kernel='rbf', C=1.0, random_state=42)
            y_pred = cross_val_predict(svm_classifier, latent_features, y, cv=skf)
            accuracy = accuracy_score(y, y_pred)
            
            results.append({
                'params': params,
                'accuracy': accuracy,
                'latent_features': latent_features,
                'y_pred': y_pred
            })
            
            print(f"精度: {accuracy:.4f}")
            
        except Exception as e:
            print(f"エラー: {e}")
            results.append({
                'params': params,
                'accuracy': 0.0,
                'error': str(e)
            })
    
    return results

# パラメータ組み合わせの定義
if 'X' in globals() and 'y' in globals():
    param_combinations = [
        {'hidden_size': 128, 'latent_size': 16, 'epochs': 20},
        {'hidden_size': 256, 'latent_size': 32, 'epochs': 20},
        {'hidden_size': 512, 'latent_size': 64, 'epochs': 20},
        {'hidden_size': 256, 'latent_size': 16, 'epochs': 20},
        {'hidden_size': 256, 'latent_size': 48, 'epochs': 20},
    ]
    
    print("パラメータ比較実験を開始します...")
    print("注意: この実験には時間がかかります")
    
    # 実験実行（コメントアウト状態で提供）
    # comparison_results = compare_vae_parameters(X, y, param_combinations, cv_folds=3)
    
    print("\n実験を実行するには上の行のコメントアウトを外してください")
else:
    print("まずデータを読み込んでください")

In [ ]:
# パラメータ比較結果の可視化
def visualize_parameter_comparison(results):
    """
    パラメータ比較結果を可視化
    """
    if not results:
        print("比較結果がありません")
        return
    
    # 結果データの整理
    param_labels = []
    accuracies = []
    
    for result in results:
        if 'error' not in result:
            params = result['params']
            label = f"H{params['hidden_size']}_L{params['latent_size']}"
            param_labels.append(label)
            accuracies.append(result['accuracy'])
    
    if not accuracies:
        print("有効な結果がありません")
        return
    
    # 棒グラフで可視化
    plt.figure(figsize=(12, 6))
    bars = plt.bar(range(len(param_labels)), accuracies, alpha=0.7, color='steelblue')
    plt.xlabel('パラメータ組み合わせ')
    plt.ylabel('交差検証精度')
    plt.title('VAEパラメータ別性能比較')
    plt.xticks(range(len(param_labels)), param_labels, rotation=45)
    plt.ylim(0, 1)
    plt.grid(True, alpha=0.3)
    
    # 数値を棒グラフ上に表示
    for bar, acc in zip(bars, accuracies):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{acc:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # 最良のパラメータを表示
    best_idx = np.argmax(accuracies)
    best_result = results[best_idx]
    print(f"\n最良のパラメータ: {best_result['params']}")
    print(f"最高精度: {best_result['accuracy']:.4f}")
    
    return best_result

# 比較結果可視化（実験実行後に使用）
if 'comparison_results' in globals():
    best_result = visualize_parameter_comparison(comparison_results)
else:
    print("パラメータ比較実験を先に実行してください")